# BGE-M3 ONNX CPU FP32 Benchmark Results
## Archived Analysis — Pre-Machine Metadata Schema

This notebook is **archived legacy**. It pre-dates the `machine_metadata`
schema, so it treats the data as flat and ignores any nested metadata.

**Path**: This notebook lives in `archived/`; results are in `../results/`.

**Schema**: Plain JSONL. If a `machine_metadata` object is present it is
dropped on load — these charts only need the flat benchmark fields.

**Visualization notes**: Throughput and latency span ~3 orders of magnitude
across pipeline stages (tokenize ≈ 100k tok/s vs embedding ≈ 400 tok/s), so
those charts use **log scales** and are **faceted by dataset** to stay legible.

In [ ]:
# ── Install required packages via uv ─────────────────────────────────────────
!uv pip install pandas matplotlib numpy jupyter -q


## Setup — Import Libraries, Resolve Paths & Define Theme

Resolves results at `../results/onnx_cpu_fp32.jsonl` (relative to `archived/`)
and installs a shared matplotlib theme + consistent color palettes so every
chart below reads the same way:

- **By pipeline stage** — tokenize (amber) / embedding (blue) / end-to-end (green)
- **By dataset** — en / th / mixed
- **By sequence length** — `max_length` 64 / 128

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display

# Results are in ../results/ relative to this notebook's location (archived/)
result_path = Path("../results/onnx_cpu_fp32.jsonl")
assert result_path.exists(), f"Results not found at {result_path.resolve()}"
print(f"Loading results from: {result_path.resolve()}")

# ── Shared visual theme ───────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "#fbfbfd",
    "axes.edgecolor":    "#cccccc",
    "axes.grid":         True,
    "grid.color":        "#e4e4ec",
    "grid.linewidth":    0.8,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.labelsize":    11,
    "font.size":         10,
    "legend.framealpha": 0.92,
})

# Consistent palettes reused by every chart below
STAGE_COLORS   = {"tokenize": "#f59e0b", "embedding": "#3b82f6", "end_to_end": "#10b981"}
DATASET_COLORS = {"en": "#3b82f6", "th": "#ef4444", "mixed": "#8b5cf6"}
LENGTH_COLORS  = {64: "#0ea5e9", 128: "#f97316"}
LENGTH_MARKERS = {64: "o", 128: "s"}

def make_run_label(df):
    """Human-readable per-run label: 'en · bs=8 · len=64'."""
    return (df["dataset"].astype(str)
            + " · bs=" + df["batch_size"].astype(str)
            + " · len=" + df["max_length"].astype(str))

def zebra_bands(ax, n, colors=("#ffffff", "#e7e7f1")):
    """Alternating horizontal background bands for n rows centered at y=0..n-1.
    Gives each run-row clear contrast so its grouped bars read as one unit."""
    for i in range(n):
        ax.axhspan(i - 0.5, i + 0.5, color=colors[i % 2], zorder=0)
    ax.set_ylim(-0.5, n - 0.5)
    ax.set_axisbelow(True)  # keep x grid above bands, below bars


## Step 1 — Load Results

Each benchmark run writes one JSON object per line (JSONL) containing:
- **Benchmark params**: `dataset`, `batch_size`, `max_length`
- **Metrics**: tokenization, embedding, and end-to-end throughput & latency

**Note**: `machine_metadata` is not present in these results (pre-v2 schema).

In [ ]:
# ── Load JSONL ───────────────────────────────────────────────────────────────
records = pd.read_json(result_path, lines=True)

# Archived analysis is flat-schema: drop the nested machine_metadata if present
if "machine_metadata" in records.columns:
    records = records.drop(columns=["machine_metadata"])

records = (records
           .sort_values(["dataset", "max_length", "batch_size"])
           .reset_index(drop=True))

print(f"Loaded {len(records)} benchmark runs")
print(f"datasets={sorted(records['dataset'].unique())}  "
      f"batch_sizes={sorted(records['batch_size'].unique())}  "
      f"max_lengths={sorted(records['max_length'].unique())}")
display(records.head(3))


## Step 2 — Performance Metrics Table

Full benchmark results sorted by `dataset → max_length → batch_size`.

| Column | Description |
|--------|-------------|
| `tokenize_tokens_per_sec` | Tokenizer throughput (tokens/sec) |
| `embedding_tokens_per_sec` | ONNX model inference throughput |
| `end_to_end_tokens_per_sec` | Tokenizer + model combined |
| `*_latency_ms_p95` | p95 latency in milliseconds |
| `*_items_per_sec` | Batches (not tokens) per second |

In [ ]:
# ── Full metrics table ────────────────────────────────────────────────────────
cols = [
    "dataset", "batch_size", "max_length", "avg_tokens_per_item",
    "tokenize_tokens_per_sec", "embedding_tokens_per_sec", "end_to_end_tokens_per_sec",
    "tokenize_items_per_sec", "embedding_items_per_sec", "end_to_end_items_per_sec",
    "tokenize_latency_ms_p95", "embedding_latency_ms_p95", "end_to_end_latency_ms_p95",
]

available_cols = [c for c in cols if c in records.columns]
display(records[available_cols].sort_values(["dataset", "max_length", "batch_size"]))


## Step 3 — Throughput by Stage (log scale, faceted by dataset)

Grouped horizontal bars comparing three throughput metrics per run:
1. **Tokenize** (amber) — raw tokenizer speed
2. **Embedding** (blue) — ONNX inference speed
3. **End-to-end** (green) — combined tokenizer + model

Tokenizer throughput (~100k tok/s) dwarfs embedding (~400 tok/s), so the
x-axis is **log-scaled** — on a linear axis the embedding/end-to-end bars
would collapse to invisible slivers.

Each run sits in its own **alternating-shade row band** with whitespace
between runs, so a run's three bars read as one group and rows are easy to
scan. One facet per dataset.

In [ ]:
# ── Throughput by stage: grouped horizontal bars, log x, faceted by dataset ───
throughput_metrics = [
    ("tokenize_tokens_per_sec",   "Tokenize",   STAGE_COLORS["tokenize"]),
    ("embedding_tokens_per_sec",  "Embedding",  STAGE_COLORS["embedding"]),
    ("end_to_end_tokens_per_sec", "End-to-end", STAGE_COLORS["end_to_end"]),
]

datasets = sorted(records["dataset"].unique())
fig, axes = plt.subplots(1, len(datasets), figsize=(6.2 * len(datasets), 6.5), sharex=True)
axes = np.atleast_1d(axes)

bar_h, step = 0.22, 0.25   # 3 bars span ±0.36 within each ±0.5 row band → clear gap between runs
for ax, ds in zip(axes, datasets):
    sub = records[records["dataset"] == ds].sort_values(["max_length", "batch_size"])
    ylabels = [f"bs={int(b)} · len={int(l)}"
               for b, l in zip(sub["batch_size"], sub["max_length"])]
    y = np.arange(len(sub))
    zebra_bands(ax, len(sub))   # alternating row bands for per-run contrast
    for i, (col, name, color) in enumerate(throughput_metrics):
        ax.barh(y + (i - 1) * step, sub[col], height=bar_h, color=color,
                label=name, alpha=0.95, edgecolor="white", linewidth=0.6, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xscale("log")
    ax.set_xlabel("tokens/sec  (log scale)")
    ax.set_title(f"dataset = {ds}")
    ax.grid(axis="x", which="both", alpha=0.4)
    ax.grid(axis="y", visible=False)

axes[0].legend(loc="lower right", fontsize=9, title="Pipeline stage")
fig.suptitle("BGE-M3 ONNX CPU FP32 — Throughput by Stage (log scale)",
             fontsize=15, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


## Step 4 — p95 Latency by Stage (log scale, faceted by dataset)

Grouped horizontal bars of three p95 latency metrics per run:
**Tokenize** (amber) / **Embedding** (blue) / **End-to-end** (green).

Tokenization (~0.5–20 ms) is dwarfed by embedding (~130 ms – 12 s at large
batch × length), so the x-axis is **log-scaled** — otherwise the tokenize bar
vanishes.

Each run's three bars are tightly grouped inside its own **alternating-shade
row band**, with whitespace between runs, so adjacent runs no longer blur
together. Use this to confirm inference dominates total latency in every run.

In [ ]:
# ── p95 latency by stage: grouped horizontal bars, log x, faceted by dataset ──
latency_metrics = [
    ("tokenize_latency_ms_p95",   "Tokenize",   STAGE_COLORS["tokenize"]),
    ("embedding_latency_ms_p95",  "Embedding",  STAGE_COLORS["embedding"]),
    ("end_to_end_latency_ms_p95", "End-to-end", STAGE_COLORS["end_to_end"]),
]

fig, axes = plt.subplots(1, len(datasets), figsize=(6.2 * len(datasets), 6.5), sharex=True)
axes = np.atleast_1d(axes)

bar_h, step = 0.22, 0.25   # 3 bars span ±0.36 within each ±0.5 row band → clear gap between runs
for ax, ds in zip(axes, datasets):
    sub = records[records["dataset"] == ds].sort_values(["max_length", "batch_size"])
    ylabels = [f"bs={int(b)} · len={int(l)}"
               for b, l in zip(sub["batch_size"], sub["max_length"])]
    y = np.arange(len(sub))
    zebra_bands(ax, len(sub))   # alternating row bands so each run's 3 bars stay visually grouped
    for i, (col, name, color) in enumerate(latency_metrics):
        ax.barh(y + (i - 1) * step, sub[col], height=bar_h, color=color,
                label=name, alpha=0.95, edgecolor="white", linewidth=0.6, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xscale("log")
    ax.set_xlabel("p95 latency ms  (log scale)")
    ax.set_title(f"dataset = {ds}")
    ax.grid(axis="x", which="both", alpha=0.4)
    ax.grid(axis="y", visible=False)

axes[0].legend(loc="lower right", fontsize=9, title="Pipeline stage")
fig.suptitle("BGE-M3 ONNX CPU FP32 — p95 Latency by Stage (log scale)",
             fontsize=15, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


## Step 5 — Tokenization Overhead Ratio

`ratio = tokenize_latency_ms_p95 / end_to_end_latency_ms_p95`

What fraction of total latency is tokenization overhead. Bars are sorted
ascending, **colored by dataset**, and the x-axis is **log-scaled** because the
ratio is consistently tiny (well under a few percent) — tokenization is
negligible against ONNX inference. Each bar is annotated with its exact percent.

In [ ]:
# ── Tokenization overhead ratio (sorted, dataset-colored, log x) ──────────────
records["tokenization_overhead_ratio"] = (
    records["tokenize_latency_ms_p95"] / records["end_to_end_latency_ms_p95"]
)

plot_df = records.sort_values("tokenization_overhead_ratio").reset_index(drop=True)
colors  = [DATASET_COLORS.get(d, "#888888") for d in plot_df["dataset"]]

fig, ax = plt.subplots(figsize=(12, 9))
y = np.arange(len(plot_df))
ax.barh(y, plot_df["tokenization_overhead_ratio"], color=colors, alpha=0.88,
        edgecolor="white", linewidth=0.5)
ax.set_yticks(y)
ax.set_yticklabels(make_run_label(plot_df), fontsize=8.5)
ax.set_xscale("log")
ax.set_xlabel("ratio  (tokenize p95 ÷ end-to-end p95, log scale)")
ax.set_title("BGE-M3 ONNX CPU FP32 — Tokenization Overhead Ratio\n"
             "tokenization is a tiny fraction of end-to-end latency (note log axis)")

for yi, v in zip(y, plot_df["tokenization_overhead_ratio"]):
    ax.text(v * 1.06, yi, f"{v:.2%}", va="center", fontsize=8, color="#374151")

ax.set_xlim(plot_df["tokenization_overhead_ratio"].min() * 0.6,
            plot_df["tokenization_overhead_ratio"].max() * 2.2)
ax.legend(handles=[Patch(facecolor=DATASET_COLORS[d], label=d)
                   for d in sorted(records["dataset"].unique())],
          loc="lower right", title="dataset")
ax.grid(axis="x", which="both", alpha=0.4)
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()


## Step 6 — Grouped Summary Table

Aggregates results grouped by benchmark params: dataset, batch_size, max_length.
Metrics aggregated by **median** (robust to outliers).

| Column | Description |
|--------|-------------|
| `tokenize_tokens_per_sec` | Median tokenizer throughput |
| `embedding_tokens_per_sec` | Median ONNX inference throughput |
| `end_to_end_tokens_per_sec` | Median combined throughput |
| `end_to_end_latency_ms_p95` | Median end-to-end p95 latency |

In [ ]:
# ── Grouped summary ────────────────────────────────────────────────────────────
group_cols = ["dataset", "batch_size", "max_length"]
available_group_cols = [c for c in group_cols if c in records.columns]

summary = (
    records.groupby(available_group_cols, as_index=False, dropna=False)
      .agg(
          tokenize_tokens_per_sec=("tokenize_tokens_per_sec", "median"),
          embedding_tokens_per_sec=("embedding_tokens_per_sec", "median"),
          end_to_end_tokens_per_sec=("end_to_end_tokens_per_sec", "median"),
          end_to_end_latency_ms_p95=("end_to_end_latency_ms_p95", "median"),
      )
)

display(summary)


## Step 7 — Batch Scaling (faceted by dataset)

One facet per dataset, sharing a y-axis so datasets are directly comparable.
Each facet plots `embedding_tokens_per_sec` vs `batch_size`, one line per
`max_length` (marker ○ = len 64, ■ = len 128), with each point value-labeled.

Use this to find the optimal batch size for a given sequence length and
language mix (en / th / mixed).

In [ ]:
# ── Batch scaling: embedding throughput vs batch size, faceted by dataset ─────
batch_vals = sorted(records["batch_size"].unique())

fig, axes = plt.subplots(1, len(datasets), figsize=(6.2 * len(datasets), 6),
                         sharey=True)
axes = np.atleast_1d(axes)

for ax, ds in zip(axes, datasets):
    sub = records[records["dataset"] == ds]
    for ml in sorted(sub["max_length"].unique()):
        s = sub[sub["max_length"] == ml].sort_values("batch_size")
        color = LENGTH_COLORS.get(ml, "gray")
        ax.plot(s["batch_size"], s["embedding_tokens_per_sec"],
                marker=LENGTH_MARKERS.get(ml, "o"), markersize=9, linewidth=2.5,
                color=color, label=f"len={ml}", alpha=0.92)
        for _, r in s.iterrows():
            ax.annotate(f"{r['embedding_tokens_per_sec']:.0f}",
                        (r["batch_size"], r["embedding_tokens_per_sec"]),
                        textcoords="offset points", xytext=(0, 9),
                        ha="center", fontsize=8, color=color, fontweight="bold")
    ax.set_title(f"dataset = {ds}")
    ax.set_xlabel("batch_size")
    ax.set_xticks(batch_vals)
    ax.grid(alpha=0.4)
    ax.legend(title="seq length", loc="best", fontsize=9)

axes[0].set_ylabel("embedding tokens/sec")
fig.suptitle("BGE-M3 ONNX CPU FP32 — Embedding Throughput vs Batch Size",
             fontsize=15, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


## Step 8 — Tokenization Efficiency Ratio

`efficiency = tokenize_tokens_per_sec / embedding_tokens_per_sec`

How many times faster the tokenizer is than the model:
- **ratio ≈ 1×** → tokenizer and model are balanced
- **ratio ≫ 1×** → model inference is the bottleneck (the case here)

Bars are sorted, **colored by dataset**, on a **log x-axis** (the ratio spans
~100× to ~2000×), with a dashed `1×` balance reference and per-bar labels.

In [ ]:
# ── Tokenization efficiency ratio (sorted, dataset-colored, log x) ────────────
records["tokenization_efficiency_ratio"] = (
    records["tokenize_tokens_per_sec"] / records["embedding_tokens_per_sec"]
)

plot_df = records.sort_values("tokenization_efficiency_ratio").reset_index(drop=True)
colors  = [DATASET_COLORS.get(d, "#888888") for d in plot_df["dataset"]]

fig, ax = plt.subplots(figsize=(12, 9))
y = np.arange(len(plot_df))
ax.barh(y, plot_df["tokenization_efficiency_ratio"], color=colors, alpha=0.88,
        edgecolor="white", linewidth=0.5)
ax.set_yticks(y)
ax.set_yticklabels(make_run_label(plot_df), fontsize=8.5)
ax.set_xscale("log")
ax.axvline(1.0, color="#16a34a", linestyle="--", linewidth=1.5, alpha=0.85)
ax.set_xlabel("ratio  (tokenize tok/s ÷ embedding tok/s, log scale)")
ax.set_title("BGE-M3 ONNX CPU FP32 — Tokenization Efficiency Ratio\n"
             "≫ 1× → tokenizer vastly outpaces the model; inference is the bottleneck")

for yi, v in zip(y, plot_df["tokenization_efficiency_ratio"]):
    ax.text(v * 1.04, yi, f"{v:.0f}×", va="center", fontsize=8, color="#374151")

ax.set_xlim(0.8, plot_df["tokenization_efficiency_ratio"].max() * 1.7)
ax.legend(handles=[Patch(facecolor=DATASET_COLORS[d], label=d)
                   for d in sorted(records["dataset"].unique())]
                  + [plt.Line2D([0], [0], color="#16a34a", linestyle="--", label="balanced (1×)")],
          loc="lower right", title="dataset")
ax.grid(axis="x", which="both", alpha=0.4)
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()
